In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def train_val_split(X, y, val_ratio=0.2, seed=42):
    np.random.seed(seed)
    indices = np.random.permutation(len(X))
    split = int(len(X) * (1 - val_ratio))

    train_idx = indices[:split]
    val_idx = indices[split:]

    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

def save_submission(predictions, filename="Polynomial_Pred.csv", column_name="target"):
    predictions = predictions.reshape(-1)
    df = pd.DataFrame({column_name: predictions})
    df.to_csv(filename, index=False)
    print(f"Submission file saved as '{filename}'")

def load_train_data(path):
    df = pd.read_csv(path)
    df = df.apply(pd.to_numeric, errors="coerce") #To avoid NAN error; It converts empty space to nan
    df = df.dropna() # This drops the NAN data

    X = df.iloc[:, :-1].values.astype(float)
    y = df.iloc[:, -1].values.astype(float)
    return X, y


def load_test_data(path):
    df = pd.read_csv(path)
    df = df.apply(pd.to_numeric, errors="coerce") 
    df = df.dropna()
    return df.values.astype(float)

def normalize_train(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1
    X_norm = (X - mu) / sigma #z-score normalization
    return X_norm, mu, sigma

def normalize_test(X, mu, sigma):
    return (X - mu) / sigma

def polynomial_features(X, degree):
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    return np.hstack([X ** i for i in range(1, degree + 1)])

def compute_cost(X, y, w, b):
    m = X.shape[0]
    errors = (X @ w + b) - y
    return (errors @ errors) / (2 * m)

def compute_gradient(X, y, w, b):
    m = X.shape[0]
    errors = (X @ w + b) - y
    dj_dw = (X.T @ errors) / m
    dj_db = np.sum(errors) / m
    return dj_db, dj_dw

def polynomial_regression(X, y, degree=2, learning_rate=0.001, num_iters=1000):
    X_norm, mu, sigma = normalize_train(X)
    X_poly = polynomial_features(X_norm, degree)

    w = np.zeros(X_poly.shape[1])
    b = 0.0
    J_history = []

    for i in range(num_iters):
        dj_db, dj_dw = compute_gradient(X_poly, y, w, b)

        w -= learning_rate * dj_dw
        b -= learning_rate * dj_db

        if i % 50 == 0:
            cost = compute_cost(X_poly, y, w, b)
            J_history.append(cost)
            print(f"Iteration {i:4d}: Cost {cost:.6f}")

    return w, b, mu, sigma, J_history

def r2_score(y_true, y_pred):
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    return 1 - (ss_res / ss_tot)

def predict(X, w, b, mu, sigma, degree):
    X_norm = normalize_test(X, mu, sigma)
    X_poly = polynomial_features(X_norm, degree)
    return X_poly @ w + b

#To run the model
X, y= load_train_data("poly_train.csv")
X_test = load_test_data("poly_test.csv")
X_train, X_val, y_train, y_val = train_val_split(X, y)
degree = 2
learning_rate = 0.001
num_iters = 1000
w, b, mu, sigma, J_history = polynomial_regression(X_train, y_train,degree=degree,learning_rate=learning_rate,num_iters=num_iters)
y_pred = predict(X_test, w, b, mu, sigma, degree)
print(y_pred[:5])

#To calculate r2 score
y_val_pred = predict(X_val, w, b, mu, sigma, degree)
r2_val = r2_score(y_val, y_val_pred)
print(f"Validation R2 Score: {r2_val:.4f}")

# Prediction file
save_submission(
    predictions=y_pred,
    filename="Polynomial_pred.csv",
    column_name="target"  
)

Iteration    0: Cost 2034390704494007955798565661436995539361058088092640317415853042271577020422946816.000000
Iteration   50: Cost 1511891105740385092849708284709225271965513590910790179584227786901068988859547648.000000
Iteration  100: Cost 1222659414311530132569206521046319089441436577727265145971255836453825027613130752.000000
Iteration  150: Cost 1038621149849472528958841323470157304861057111375177079732849828054133798198050816.000000
Iteration  200: Cost 907736205648993573465551766114531799305217839859244206006892814277168149206925312.000000
Iteration  250: Cost 807720534224990982862756010630320482022208865606362869564369170734337686532784128.000000
Iteration  300: Cost 728148402767005982284526044315567243670670510406774806094787465635953559094689792.000000
Iteration  350: Cost 663491129475416034838154069899597134502954928174377041725178319963086489558974464.000000
Iteration  400: Cost 610377528989448320553054465382464505887377701901963217159793471609295055766945792.000000
Iterat